# Atividade do dia: Segmentação de clientes com algoritmos de cluster não supervisionados

#### Objetivo: Realizar uma análise de mercado e criar um algoritmo de cluster para segmentar clientes. Ao final os resultados terão sua própria análise.

# Resumo

Este notebook explorou a segmentação de clientes utilizando algoritmos de cluster não supervisionados. Realizamos uma Análise Exploratória de Dados (EDA) abrangente para entender as características dos clientes e as relações entre as variáveis. Identificamos a presença de valores nulos e inconsistências nos dados, que foram tratados durante o pré-processamento. Novas features foram criadas para enriquecer o conjunto de dados.

Em seguida, avaliamos a dimensionalidade dos dados utilizando PCA e Kernel PCA. Os resultados indicaram que os dados não são linearmente separáveis, e o Kernel PCA com kernel 'sigmoid' apresentou melhor desempenho na preservação da variância.

Experimentamos dois algoritmos de clustering: DBSCAN e K-Means. O DBSCAN, mesmo com parâmetros otimizados, identificou a maioria dos dados como pertencentes a um único cluster grande com muitos outliers, o que não se alinha com o objetivo de segmentação em grupos distintos. O K-Means, configurado para 4 clusters (correspondendo ao número de segmentos originais), gerou clusters que, embora não replicassem perfeitamente os segmentos originais, forneceram uma base para a análise de diferentes perfis de clientes.

A comparação entre os segmentos originais e os clusters gerados pelo K-Means revelou diferenças nas distribuições de variáveis como idade, tamanho da família, experiência de trabalho, score de gastos, profissão e estado civil. Isso sugere que o K-Means encontrou uma estrutura de agrupamento diferente da segmentação pré-existente.

Por fim, integramos o MLflow para rastrear os experimentos, registrando parâmetros, métricas (como o silhouette score) e os modelos treinados (Kernel PCA, DBSCAN e K-Means) para futuras análises e comparações.

# Configurações Iniciais

## Bibliotecas

In [1]:
#%pip install -U -q plotly seaborn
#%restart_python

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
import plotly.graph_objects as go

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.impute import SimpleImputer, KNNImputer

from sklearn.decomposition import PCA, KernelPCA
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

## Configuração do Plotly

In [3]:
pio.templates["dark_elegance_premium"] = pio.templates["plotly_dark"]

pio.templates["dark_elegance_premium"].layout.update(
    {
        "font": {
            "family": "Inter, Helvetica Neue, Arial",
            "size": 14,
            "color": "#EAEAEA",
        },
        "paper_bgcolor": "#0E1117",
        "plot_bgcolor": "#0E1117",
        "title": {
            "x": 0.5,
            "xanchor": "center",
            "font": {"size": 28, "color": "#FFFFFF"},
        },
        "xaxis": {
            "gridcolor": "#1C1F26",
            "zeroline": False,
            "showline": True,
            "linecolor": "#3A3A3A",
            "ticks": "outside",
            "tickcolor": "#888",
            "title_font": {"color": "#CCCCCC"},
        },
        "yaxis": {
            "gridcolor": "#1C1F26",
            "zeroline": False,
            "showline": True,
            "linecolor": "#3A3A3A",
            "ticks": "outside",
            "tickcolor": "#888",
            "title_font": {"color": "#CCCCCC"},
        },
        "legend": {
            "bgcolor": "rgba(0,0,0,0)",
            "bordercolor": "rgba(255,255,255,0.1)",
            "borderwidth": 1,
            "font": {"color": "#EAEAEA"},
            "orientation": "h",
            "x": 0.5,
            "xanchor": "center",
            "y": -0.40,
        },
        "coloraxis_colorbar": {
            "tickcolor": "#EAEAEA",
            "title": {"font": {"color": "#EAEAEA"}},
        },
    }
)

pio.templates.default = "dark_elegance_premium"

# Coleta de Dados

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Dataset de Treino

In [5]:
train = pd.read_csv("/content/drive/MyDrive/MBA em Inteligência Artificial com Machine Learning para Negócios/Aula - 21/Atividade/Train.csv")
#spark_df_train = spark.read.table("workspace.default.train")
#train = spark_df_train.toPandas()
#display(train)

## Dataset de Teste

In [6]:
test = pd.read_csv("/content/drive/MyDrive/MBA em Inteligência Artificial com Machine Learning para Negócios/Aula - 21/Atividade/Test.csv")
#spark_df_test = spark.read.table("workspace.default.test")
#test = spark_df_test.toPandas()
#display(test)

# Análise Exploratória de Dados

## Objetivo
**Entender a relação entre as variáveis, focando em como elas podem ajudar a separar cada cliente em um segmento.**

## Sumário
**Observação:** A primeira versão desse notebook utilizava `dbutils.data.summarize(df)`, mas o período de teste grátis do Databricks acabou e com ele se foi essa funcionalidade. O texto será mantido o mesmo, mas tentarei criar a mesma célula que trazia os dados do sumário, com exceção das distribuições, que serão plotadas depois.

In [7]:
def summarize_df_pandas(df: pd.DataFrame) -> pd.DataFrame:
    """
    Gera um resumo estatístico do DataFrame similar ao dbutils.data.summarize(),
    mostrando tipo, contagem de nulos, distintos, estatísticas descritivas e valor mais frequente.
    """
    total = len(df)
    summaries = []

    for col in df.columns:
        s = df[col]
        dtype = str(s.dtype)
        nulls = s.isna().sum()
        non_nulls = total - nulls
        distinct = s.nunique(dropna=True)
        nulls_percent = round((nulls / total) * 100, 2) if total > 0 else np.nan

        summary = {
            "column": col,
            "type": dtype,
            "non_nulls": non_nulls,
            "nulls_percent": nulls_percent,
            "distinct_count": distinct,
            "mean": None,
            "stddev": None,
            "min": None,
            "25%": None,
            "50%": None,
            "75%": None,
            "max": None,
            "top_value": None,
            "top_freq": None,
        }

        if pd.api.types.is_numeric_dtype(s):
            desc = s.describe(percentiles=[0.25, 0.5, 0.75])
            summary.update(
                {
                    "mean": round(desc.get("mean", np.nan), 2),
                    "stddev": round(desc.get("std", np.nan), 2),
                    "min": desc.get("min", np.nan),
                    "25%": desc.get("25%", np.nan),
                    "50%": desc.get("50%", np.nan),
                    "75%": desc.get("75%", np.nan),
                    "max": desc.get("max", np.nan),
                }
            )
        else:
            vc = s.value_counts(dropna=True)
            if not vc.empty:
                top = vc.index[0]
                freq = vc.iloc[0]
                summary.update({"top_value": top, "top_freq": freq})

        summaries.append(summary)

    df_summary = pd.DataFrame(summaries)
    df_summary = df_summary.sort_values(
        by=["type", "column"], ascending=[True, True]
    ).reset_index(drop=True)
    return df_summary

In [8]:
summary_train = summarize_df_pandas(train)
display(summary_train)

,column,type,non_nulls,nulls_percent,distinct_count,mean,stddev,min,25%,50%,75%,max,top_value,top_freq
0,Family_Size,float64,7733,4.15,9,2.85,1.53,1.0,2.00,3.0,4.00,9.0,None,NaN
1,Work_Experience,float64,7239,10.28,15,2.64,3.41,0.0,0.00,1.0,4.00,14.0,None,NaN
2,Age,int64,8068,0.00,67,43.47,16.71,18.0,30.00,40.0,53.00,89.0,None,NaN
3,ID,int64,8068,0.00,8068,463479.21,2595.38,458982.0,461240.75,463472.5,465744.25,467974.0,None,NaN
4,Ever_Married,object,7928,1.74,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,4643.0
5,Gender,object,8068,0.00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Male,4417.0
6,Graduated,object,7990,0.97,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,4968.0
7,Profession,object,7944,1.54,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Artist,2516.0
8,Segmentation,object,8068,0.00,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,D,2268.0
9,Spending_Score,object,8068,0.00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low,4878.0


Com o sumário, podemos ver que **nos dados de treinamento** (train):
- Existem 2 colunas do tipo *int*, 2 colunas *float* e 7 colunas *string*. Então temos **4 colunas numéricas e 7 colunas categóricas**;
- A contagem de valores é **8.068**.

**Colunas Numéricas:**

- Algumas colunas possuem **valores nulos**. `Work_Experience` é a que mais tem nulos com cerca de **10.28% de valores faltando**, além de que é a **única coluna com o valor 0** em algumas linhas (32.02% das linhas);
- Se desconsiderarmos `ID`, já que ele é simplesmente a identificação de cada pessoa, a **média e o desvio padrão das colunas numéricas estão com valores baixos**, o que é bom. Vale a pena destacar que `Age` tem a média e desvio padrão um pouco mais altos, isso faz sentido quando olhamos para a distribuição da idade, que tem uma **calda para a direita**.

**Colunas Categóricas:**
- 4 colunas possuem uma pequena porcentagem de valores faltando (máximo de 1.74%);
- As quantidades e valores únicos de cada coluna são:
    - `Ever_Married`: 2 (*Yes*, *No*)
    - `Graduated`: 2 (*Yes*, *No*)
    - `Profession`: 9 (*Artist*, *Healthcare*, *Entertrainment*, *Engineer*, *Doctor*, *Lawyer*, *Executive*, *Marketing*, *Homemaker*)
    - `Spending_Score`: 3 (*Low*, *Average*, *High*)
    - `Var_1`: 6 (*Cat_1*, *Cat_2*, *Cat_3*, *Cat_4*, *Cat_5*, *Cat_6*)
    - `Segmentation`: 4 (*A*, *B*, *C*, *D*)
- Olhando as distribuições, podemos ver que `Profession`, `Spending_Score` e `Var_1` tem caldas para a direita. As outras colunas estão quase totalmente balanceadas.

In [9]:
summary_test = summarize_df_pandas(test)
display(summary_test)

,column,type,non_nulls,nulls_percent,distinct_count,mean,stddev,min,25%,50%,75%,max,top_value,top_freq
0,Family_Size,float64,2514,4.30,9,2.83,1.55,1.0,2.0,2.0,4.0,9.0,None,NaN
1,Work_Experience,float64,2358,10.24,15,2.55,3.34,0.0,0.0,1.0,4.0,14.0,None,NaN
2,Age,int64,2627,0.00,67,43.65,16.97,18.0,30.0,41.0,53.0,89.0,None,NaN
3,ID,int64,2627,0.00,2627,463433.92,2618.25,458989.0,461162.5,463379.0,465696.0,467968.0,None,NaN
4,Ever_Married,object,2577,1.90,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,1520.0
5,Gender,object,2627,0.00,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Male,1424.0
6,Graduated,object,2603,0.91,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Yes,1602.0
7,Profession,object,2589,1.45,9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Artist,802.0
8,Spending_Score,object,2627,0.00,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Low,1616.0
9,Var_1,object,2595,1.22,7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cat_6,1672.0


O sumário dos dados de teste mostra que eles são parecidos com os dados de treino.

## Distribuição dos Dados
Para entender como os dados estão distribuídos, criei esse gráfico que permite olhar cada distribuição separada, basta utilizar o menu lateral para escolher a distribuição desejada. *Caso o menu não esteja responsivo, basta rodar novamente a célula.*

In [10]:
cols = sorted(train.columns.tolist())
initial_col = cols[0]

fig = px.histogram(
    data_frame=train,
    x=initial_col,
    opacity=0.7,
    marginal="box",
    title=f'Distribuição de "{initial_col}"',
    text_auto=True,
)

fig.update_layout(yaxis_title="Contagem")
fig.update_traces(marker_line_color="black", marker_line_width=1)
fig.data[0].hovertemplate = (
    f"<b>{initial_col}:</b> %{{x}}<br><b>Contagem:</b> %{{y}}<extra></extra>"
)

buttons = []
for col in cols:
    is_numeric = pd.api.types.is_numeric_dtype(train[col])
    category_order = None if is_numeric else "category ascending"
    hist_hover = f"<b>{col}:</b> %{{x}}<br><b>Contagem:</b> %{{y}}<extra></extra>"
    box_hover = f"<b>{col}</b><br>%{{x}}<extra></extra>"
    buttons.append(
        {
            "method": "update",
            "label": col,
            "args": [
                {
                    "x": [train[col], train[col]],
                    "visible": [True, is_numeric],
                    "hovertemplate": [hist_hover, box_hover],
                },
                {
                    "xaxis.title.text": col,
                    "title.text": f'Distribuição de "{col}"',
                    "xaxis.categoryorder": category_order,
                    "yaxis2.visible": is_numeric,
                    "xaxis2.visible": is_numeric,
                },
            ],
        }
    )

fig.update_layout(
    updatemenus=[
        {
            "buttons": buttons,
            "direction": "down",
            "showactive": True,
            "x": 1.01,
            "xanchor": "left",
            "y": 1.15,
            "yanchor": "top",
        }
    ],
    annotations=[
        dict(
            text="Selecione a Variável",
            showarrow=False,
            x=1.15,
            y=1.25,
            xref="paper",
            yref="paper",
            align="left",
            font={"color": "#EAEAEA", "size": 14},
        )
    ],
)


fig.show()

# Questionamentos Sobre os Dados

Já que o objetivo é separar os clientes por segmento, vou realizar uma análise focando nessa variável (Segmentation), mas também terá foco nas relações entre as variáveis.

## Existem Quantos Clientes por Segmento?

In [11]:
df_counts = train["Segmentation"].value_counts().reset_index()
df_counts.columns = ["Segmentation", "Contagem"]
df_counts["Porcentagem"] = df_counts["Contagem"] / 8068

fig = px.bar(
    df_counts,
    y="Segmentation",
    x="Contagem",
    title="Quantidade de Clientes por Segmento",
    labels={"Contagem": "Contagem", "Segmentation": "Segmento"},
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    custom_data=["Porcentagem"],
)

fig.update_traces(
    texttemplate="%{x} (%{customdata[0]:.2%})",
    textposition="auto",
    hovertemplate="<b>Segmento:</b> %{y}<br>"
    + "<b>Contagem:</b> %{x}<br>"
    + "<b>Porcentagem:</b> %{customdata[0]:.2%}"
    + "<extra></extra>",
)

fig.update_layout(xaxis_title="Contagem", xaxis_range=[0, 8070])
fig.add_vline(x=8068, line_dash="dash", line_color="white", line_width=2)
fig.add_annotation(
    y=0,
    x=8068,
    text="Total de valores no dataset: 8.068",
    showarrow=False,
    xshift=10,
    align="center",
    yanchor="bottom",
    textangle=90,
    font=dict(size=12),
)

fig.show()

- O segmento A possui 1972 clientes;
- O segmento B possui 1858 clientes;
- O segmento C possui 1970 clientes;
- O segmento D possui 2268 clientes;

Em média, temos 2017 clientes por segmento.

## Qual é a idade dos clientes em cada segmento?

In [12]:
fig = px.box(
    train,
    x="Segmentation",
    y="Age",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    title="Idade por Segmento",
)

fig.update_layout(
    xaxis_title="Segmento", yaxis_title="Idade", yaxis_dtick=10, showlegend=False
)

fig.show()

Se considerarmos uma análise por grupos de mais jovens, indo até mais velhos, o *Segmento D* tem **a maior quantidade de jovens**, com sua **maioria entre 22 e 38 anos** mas sendo aceitável considerar um grupo de **clientes de 18 até 68 anos** e com alguns **outliers** que vão de 63 até 89 anos;

Logo após, o *Segmento A* é o grupo  que em sí tem **maioria entre 33 e 52 anos**, sendo aceitável considerar um grupo de **clientes de 18 até 80 anos** e com **outliers** que vão de 81 até 89 anos;

Já os *Segmentos B e C*, são bem parecidos. Ambos abrangem um grupo de **idade entre 18 e 89 anos**, a única diferença é que a maioria das idades respectivamente é **37 até 58 anos** e **38 até 59 anos** para os segmentos B e C.

Isso mostra que os **segmentos B e C tem um perfil de clientes com idades parecidas** e que apesar dos **segmentos A e D terem um perfil de clientes com menos idade**, eles também tem outliers, que são clientes que **preferem os segmentos A ou D ao invés do B ou C**.



## Qual é o Tamanho da Família dos Clientes em Cada Segmento?

In [13]:
fig = px.box(
    train,
    x="Segmentation",
    y="Family_Size",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    title="Tamanho da Família por Segmento",
)

fig.update_layout(
    xaxis_title="Segmento",
    yaxis_title="Tamanho da Família",
    yaxis_dtick=1,
    showlegend=False,
)

fig.show()

Os segmentos B, C e D são **quase idênticos**, todos tem sua maioria entre **2 e 4 familiares** mas podendo ir de **1 até 7 familiares** e com alguns outliers que tem **8 ou 9 familiares**.

Já os clientes do segmento A tem menos familiares, com sua maioria entre **1 e 3 familiares**, sendo aceitável considerar entre **1 e 6 familiares** e com outliers que vão de **7 até 9 familiares**.

Isso mostra que as pessoas do segmento A tem menos familiares do que os demais.

In [14]:
fig = px.box(
    train,
    x="Segmentation",
    y="Work_Experience",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    title="Experiência de Trabalho por Segmento",
)

fig.update_layout(
    xaxis_title="Segmento",
    yaxis_title="Experiência de Trabalho (anos)",
    yaxis_dtick=1,
    showlegend=False,
)

fig.show()

De cara, é possível notar que o segmento que mais tem experiência de trabalho é o D, mas ao mesmo tempo ele é o segmento com menos idade entre os 4. **Isso deve ser observado depois**. Ele tem a maioria dos valores entre **0 e 6 anos de experiência** mas incrivelmente tem todos os valores como aceitáveis;

O segmento com **menos experiência de trabalho é o C**, com a maioria entre **0 e 3 anos**, sendo aceitável **até 7 anos** de experiência e com **outliers até 14 anos**;

O segmento B tem a maioria entre **0 e 4 anos**, sendo aceitável **até 10 anos** de experiência e com **outliers até 14 anos**;

O segmento A tem a maioria entre **0 e 5 anos**, sendo aceitável até **12 anos** de experiência e com **outliers até 14 anos**;

Vamos analisar a densidade da idade pela experiência de trabalho, para poder entender a quantidade de pessoas jovens que trabalha:

In [15]:
fig = px.density_heatmap(
    train,
    x="Age",
    y="Work_Experience",
    color_continuous_scale="Viridis",
    nbinsx=72,
    nbinsy=15,
    text_auto=True,
    title="Densidade de Idade por Experiência de Trabalho",
)

fig.update_layout(
    xaxis_title="Idade",
    yaxis_title="Experiência de Trabalho (anos)",
    showlegend=False,
    yaxis_dtick=1,
    xaxis_dtick=10,
    coloraxis_colorbar_title_text="Contagem",
)

fig.show()

O mapa de densidade mostra que a maioria dos valores parecem estar corretos, no sentido de que eles se encaixam na regra de trabalho da nossa sociedade, ou seja, pessoas não podem ter um tempo de experiência que não faz sentido. Apesar disso, é possível ver que algumas pessoas tem 18 anos e 13 ou 14 anos de experiência, **isso é um erro de coleta de dados do dataset**, não tem como saber se essas pessoas mentiram na coleta de dados ou se é algum erro, como ao invés de submeter para a coleta o valor de 1.4 anos de experiência, foi submetido 14 anos. **Isso será tratado no pré-processamento**.

## Qual é o score de gastos de cada segmento?

In [16]:
fig = px.histogram(
    train,
    x="Spending_Score",
    color="Segmentation",
    barmode="group",
    category_orders={
        "Spending_Score": ["Low", "Average", "High"],
        "Segmentation": ["A", "B", "C", "D"],
    },
    title="Score de Gastos por Segmento",
    text_auto=True,
    labels={"Spending_Score": "Score de Gastos", "Segmentation": "Segmento"},
)

fig.update_layout(
    xaxis_title="Score de Gastos",
    yaxis_title="Contagem",
)

fig.show()

A maioria dos clientes gasta pouco, sendo que os clientes que menos gastam são os do segmento D, faz sentido, já que eles são os mais novos. Os clientes do segmento C são os que mais tem um gasto moderado, e nos gastos altos, temos poucos clientes de todos os segmentos.

## Será que os gêneros tem influência nos segmentos?

In [17]:
fig = px.histogram(
    train,
    x="Gender",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    barmode="group",
    title="Gênero por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)
fig.update_layout(xaxis_title="Segmento", yaxis_title="Gênero")
fig.show()

Eles não tem influência. Os gêneros estão bem distribuidos pelos segmentos.

In [18]:
fig = px.histogram(
    train,
    x="Profession",
    color="Segmentation",
    category_orders={
        "Segmentation": ["A", "B", "C", "D"],
    },
    barmode="group",
    title="Profissão por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)

fig.update_layout(
    xaxis_title="Profissão",
    yaxis_title="Contagem",
    xaxis_categoryorder="total descending",
)

fig.show()

A  maioria das pessoas do dataset são artistas e trabalhadores da saúde, enquanto nas outras profissões, os dados são bem parecidos. As pessoas do segmento A, B e C são artistas e a maioria das pessoas no segmento D são da área da saúde.

## Como cada profissão gasta seu dinheiro em cada segmento?

In [19]:
profession_order = train["Profession"].value_counts().index.tolist()

fig = px.density_heatmap(
    train,
    y="Profession",
    x="Spending_Score",
    facet_col="Segmentation",
    category_orders={
        "Segmentation": ["A", "B", "C", "D"],
        "Spending_Score": ["Low", "Average", "High"],
        "Profession": profession_order,
    },
    title="Densidade de Score de Gastos por Profissão e Segmento",
    color_continuous_scale="Viridis",
    text_auto=True,
)

fig.update_xaxes(title_text="Score de Gastos")

fig.update_layout(yaxis_title="Profissão")

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.update_layout(coloraxis_colorbar_title_text="Contagem")

fig.show()

Esse gráfico é interessante porque mostra como cada segmento gasta seu dinheiro. Nos segmentos A, B e C os gastos costumam ser baixos ou médios, com a exceção dos advogados e executivos, que gastam mais. Também é notável a quantidade de artistas que tem gastos baixos ou moderados. No segmento D, temos muito mais gastos baixos do que médios e altos, especialmente no ramo da saúde.

## Qual é a relação entre graduação e segmento?

In [20]:
fig = px.histogram(
    train,
    x="Graduated",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    barmode="group",
    title="Graduação por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)
fig.update_layout(xaxis_title="Graduado?", yaxis_title="Contagem")
fig.show()

Vemos um número maior de pessoas não graduadas no segmento D, enquanto temos mais pessoas graduadas nos segmentos A, B e C.

In [21]:
fig = px.histogram(
    train,
    x="Ever_Married",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    barmode="group",
    title="Estado Civil por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)
fig.update_layout(xaxis_title="Casado?", yaxis_title="Contagem")
fig.show()

Interessante notar que o gráfico de graduação se parece muito com o de casamento, isso pode indicar uma relação entre a disponibilidade de se casar e possuir uma graduação ou não.

## Qual é a relação entre o estado civil e os gastos?

In [22]:
fig = px.density_heatmap(
    train,
    x="Spending_Score",
    y="Ever_Married",
    text_auto=True,
    category_orders={"Spending_Score": ["Low", "Average", "High"]},
    title="Score de Gastos por Estado Civil",
    color_continuous_scale="Viridis",
)

fig.update_layout(
    xaxis_title="Score de Gastos",
    yaxis_title="Estado Civil",
    coloraxis_colorbar_title_text="Contagem",
)

fig.show()

Não poderia ser mais evidente que isso, pessoas solteiras tem um score de gastos baixo, enquanto pessoas casadas tem um score melhor distribuído.

In [23]:
fig = px.density_heatmap(
    train,
    x="Spending_Score",
    y="Graduated",
    text_auto=True,
    category_orders={"Spending_Score": ["Low", "Average", "High"]},
    title="Score de Gastos por Nível de Graduação",
    color_continuous_scale="Viridis",
)

fig.update_layout(
    xaxis_title="Score de Gastos",
    yaxis_title="Status de Graduação",
    coloraxis_colorbar_title_text="Contagem",
)

fig.show()

O nível de graduação também influencia em como os clientes gastam o dinheiro. Clientes que não são graduados tem um score de gastos menor, enquanto clientes que são graduados tem um score maior, apesar de que muitos também tem um score baixo.

In [24]:
fig = px.density_heatmap(
    train,
    x="Var_1",
    y="Segmentation",
    text_auto=True,
    title="Relação entre Var_1 e Segmentação",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    color_continuous_scale="Viridis",
)

fig.update_layout(
    xaxis_title="Var_1",
    yaxis_title="Segmento",
    coloraxis_colorbar_title_text="Contagem",
    xaxis_categoryorder="category ascending",
)

fig.show()

Cat_6 tem bastante representação em todos os segmentos, logo após temos Cat_2, Cat_3 e Cat_4, que tem menos valores mas que ainda sim são representativos. Cat_1, Cat_5 e Cat_7 parecem ser categorias mais exclusivas.

# Pré-processamento

Como já foi observado no EDA, o dataset tem valores nulos e valores que parecem estar errados, como a experiência de trabalho de alguns jovens. Vamos tratá-los e criar novos atributos.

In [25]:
segmentation = train["Segmentation"].copy()
train.drop(labels="Segmentation", axis=1, inplace=True)

In [26]:
# Classes de correção
class WorkExperienceFixer(BaseEstimator, TransformerMixin):
    def __init__(self, age_col="Age", work_col="Work_Experience", min_start_age=18):
        self.age_col = age_col
        self.work_col = work_col
        self.min_start_age = min_start_age

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        mask = (
            X[self.age_col].notna()
            & X[self.work_col].notna()
            & (X[self.age_col] - X[self.work_col] < self.min_start_age)
        )
        X.loc[mask, self.work_col] = 0
        return X


# Feature engineering
class FeatureEngineer(BaseEstimator, TransformerMixin):
    def __init__(self, age_col="Age", work_col="Work_Experience"):
        self.age_col = age_col
        self.work_col = work_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        # Idade que começou a trabalhar
        X["Age_Started_Work"] = X[self.age_col] - X[self.work_col]

        # Household Type
        conditions = [
            (X["Ever_Married"] == "No") & (X["Family_Size"] == 1),
            (X["Ever_Married"] == "Yes") & (X["Family_Size"] == 2),
            (X["Ever_Married"] == "Yes") & (X["Family_Size"] > 2),
            (X["Ever_Married"] == "No") & (X["Family_Size"] > 1),
        ]
        choices = ["Solteiro", "Casal", "Familia_Nuclear", "Familia_Outros"]
        X["Household_Type"] = np.select(conditions, choices, default="Familia_Outros")

        # Agrupamento de profissões
        profession_map = {
            "Doctor": "Alta_Renda",
            "Lawyer": "Alta_Renda",
            "Engineer": "Alta_Renda",
            "Executive": "Alta_Renda",
            "Artist": "Criativo",
            "Entertainment": "Criativo",
            "Healthcare": "Servicos",
            "Marketing": "Servicos",
            "Homemaker": "Casa",
        }
        X["Profession_Group"] = X["Profession"].map(profession_map).fillna("Outros")

        return X


# Função para criar pipeline
def create_pipeline(
    numerical_cols,
    categorical_nominal_cols,
    categorical_ordinal_cols,
    ordinal_categories,
):

    # Pipeline numérico (RandomForestRegressor)
    numeric_pipeline = Pipeline(
        [("imputer",IterativeImputer(
                        estimator=RandomForestRegressor(
                            n_estimators=100, random_state=42, n_jobs=-1
                        ),
                        max_iter=10,
                        initial_strategy="median",
                    ),
                ),
            ("scaler", StandardScaler()),
        ]
    )

    # Pipeline categórico nominal (RandomForestClassifier)
    # Codificamos temporariamente com OrdinalEncoder para imputação
    categorical_nominal_pipeline = Pipeline(
        [
            ("ordinal_encode_temp", OrdinalEncoder()),
            ("imputer",IterativeImputer(
                        estimator=RandomForestClassifier(n_estimators=100, random_state=42),
                        max_iter=10,
                        initial_strategy="most_frequent",
                    ),
                ),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]
    )

    # Pipeline categórico ordinal (RandomForestClassifier)
    categorical_ordinal_pipeline = Pipeline(
        [
            ("ordinal_encode_temp", OrdinalEncoder(categories=ordinal_categories)),
            ("imputer",IterativeImputer(
                        estimator=RandomForestClassifier(n_estimators=100, random_state=42),
                        max_iter=10,
                        initial_strategy="most_frequent",
                    ),
                ),
            ("scaler", StandardScaler()),
        ]
    )

    # ColumnTransformer
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipeline, numerical_cols),
            ("nominal", categorical_nominal_pipeline, categorical_nominal_cols),
            ("ordinal", categorical_ordinal_pipeline, categorical_ordinal_cols),
        ],
        remainder="drop",
    )

    # Pipeline final
    pipeline = Pipeline(
        [
            ("fix_work_exp", WorkExperienceFixer()),
            ("feature_engineering", FeatureEngineer()),
            ("preprocessor", preprocessor),
        ]
    )

    return pipeline

In [27]:
numerical_cols = train.select_dtypes(include=["int64", "float64"]).columns.tolist()
numerical_cols.remove("ID")

categorical_nominal_cols = train.select_dtypes(include=["object"]).columns.tolist()
categorical_nominal_cols.remove("Spending_Score")

categorical_ordinal_cols = ["Spending_Score"]
ordinal_categories = [["Low", "Average", "High"]]

In [28]:
# Criar pipeline
pipe = create_pipeline(
    numerical_cols,
    categorical_nominal_cols,
    categorical_ordinal_cols,
    ordinal_categories,
)

# Mostra o pipeline
display(pipe)

# Aplicar no dataset
X_train_transformed = pipe.fit_transform(train.drop(columns=["ID"]))
print(f"O novo shape do dataset é {X_train_transformed.shape}")

Pipeline(steps=[('fix_work_exp', WorkExperienceFixer()),
                ('feature_engineering', FeatureEngineer()),
                ('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   IterativeImputer(estimator=RandomForestRegressor(n_jobs=-1,
                                                                                                                    random_state=42),
                                                                                    initial_strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Work_Experience',
                                                   'Family_Size...
                                                                                 sparse_output=False))]),
                                                  ['Gender', 'Ever_Married',
                                                   'Graduated', 'Profession',
                                                   'Var_1']),
                                                 ('ordinal',
                                                  Pipeline(steps=[('ordinal_encode_temp',
                                                                   OrdinalEncoder(categories=[['Low',
                                                                                               'Average',
                                                                                               'High']])),
                                                                  ('imputer',
                                                                   IterativeImputer(estimator=RandomForestClassifier(random_state=42),
                                                                                    initial_strategy='most_frequent')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Spending_Score'])]))])

/usr/local/lib/python3.12/dist-packages/sklearn/impute/_iterative.py:895: ConvergenceWarning:

[IterativeImputer] Early stopping criterion not reached.



O novo shape do dataset é (8068, 26)


In [29]:
X_train_df = pd.DataFrame(X_train_transformed, columns=pipe[2:].get_feature_names_out())
display(X_train_df)

,num__Age,num__Work_Experience,num__Family_Size,nominal__Gender_0.0,nominal__Gender_1.0,nominal__Ever_Married_0.0,nominal__Ever_Married_1.0,nominal__Graduated_0.0,nominal__Graduated_1.0,nominal__Profession_0.0,...,nominal__Profession_7.0,nominal__Profession_8.0,nominal__Var_1_0.0,nominal__Var_1_1.0,nominal__Var_1_2.0,nominal__Var_1_3.0,nominal__Var_1_4.0,nominal__Var_1_5.0,nominal__Var_1_6.0,ordinal__Spending_Score
0,-1.284623,-0.427128,0.759522,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.736833
1,-0.327151,0.293695,0.097824,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.612411
2,1.408268,-0.427128,-1.225572,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.736833
3,1.408268,-0.755153,-0.563874,0.0,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.961655
4,-0.207467,2.105526,2.082918,1.0,0.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.961655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8063,-1.284623,-0.755153,2.744617,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.736833
8064,-0.506677,0.228921,0.759522,0.0,1.0,1.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-0.736833
8065,-0.626361,-0.427128,-1.225572,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.736833
8066,-0.985413,-0.427128,0.759522,1.0,0.0,1.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,-0.736833


In [30]:
corr_df = X_train_df.copy()

fig = px.imshow(
    corr_df.corr(),
    text_auto=True,
    title="Matriz de Correlação das Variáveis Numéricas",
    color_continuous_scale="Viridis",
    zmin=-1,
    zmax=1,
    color_continuous_midpoint=0,
)

fig.update_layout(width=1200, height=1200, title_x=0.5)

fig.update_xaxes(tickangle=45, tickfont=dict(size=10))

fig.update_yaxes(tickfont=dict(size=10))

fig.show()

Como podemos ver, temos poucas relações lineares fortes no dataset, isso é bom porque nossos dados não terão o problema da multicolinearidade.

In [31]:
X_test_transformed = pipe.transform(test.drop(columns=["ID"]))
print(f"O novo shape do dataset é {X_test_transformed.shape}")

O novo shape do dataset é (2627, 26)


# Usar PCA ou Kernel PCA?

Para checar se o PCA ou se o Kernel PCA é o melhor para o nosso caso, podemos plotar o gráfico da variância explicada do PCA e também plotar o gráfico de visualização dos componentes principais. Farei isso para o PCA e Kernel PCA.

In [32]:
pca = PCA().fit(X_train_transformed)

fig = px.line(
    np.cumsum(pca.explained_variance_ratio_),
    title="Variância explicada pelo PCA",
    labels={"index": "Número de Componentes", "value": "Variância Explicada Acumulada"},
)

fig.update_layout(showlegend=False)

fig.add_hline(y=0.90, line_dash="dash", line_color="red", annotation_text="90%")

fig.show()

In [33]:
# Definindo os kernels
kernels_to_test = ["linear", "rbf", "poly", "sigmoid"]

# Lista para armazenar os resultados de cada kernel
results_list = []

# Loop para calcular a variância para cada kernel
for kernel_name in kernels_to_test:
    kpca = KernelPCA(n_components=35, kernel=kernel_name, random_state=42)

    # Ajustar aos dados
    kpca.fit(X_train_transformed)

    # Pegar os autovalores
    eigenvalues = kpca.eigenvalues_
    explained_variance = eigenvalues / np.sum(eigenvalues)

    # Calcular a soma acumulada
    cumulative_variance = np.cumsum(explained_variance)

    # Armazenar em um DataFrame temporário
    temp_df = pd.DataFrame(
        {
            "Componente": np.arange(1, len(cumulative_variance) + 1),
            "Variancia Acumulada": cumulative_variance,
        }
    )
    temp_df["Kernel"] = kernel_name

    # Adicionar à nossa lista de resultados
    results_list.append(temp_df)

# Combinar todos os DataFrames de resultados em um só
results_df = pd.concat(results_list)

# Plotar o gráfico de linhas comparativo
fig = px.line(
    results_df,
    x="Componente",
    y="Variancia Acumulada",
    color="Kernel",
    title="Comparação da Variância Acumulada por Kernel (KPCA) - Top 35",
    labels={
        "Componente": "Número de Componentes",
        "Variancia Acumulada": "Variância Explicada Acumulada",
    },
)

# Adicionar linhas de referência
fig.add_hline(y=0.90, line_dash="dash", line_color="gray", annotation_text="90%")

fig.show()

## Graficos 2D e 3D

In [34]:
# 2 Dimensões
pca_2d = PCA(n_components=2)
X_2d = pca_2d.fit_transform(X_train_transformed)

# 3 Dimensões
pca_3d = PCA(n_components=3)
X_3d = pca_3d.fit_transform(X_train_transformed)

In [35]:
fig = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=("PCA com 2 Componentes", "PCA com 3 Componentes"),
    specs=[[{"type": "xy"}, {"type": "scene"}]],
)

fig_2d = px.scatter(
    x=X_2d[:, 0],
    y=X_2d[:, 1],
    color=segmentation,
    category_orders={"color": ["A", "B", "C", "D"]},
)

fig_2d.update_traces(marker=dict(line=dict(width=0.5, color="white")))

fig_3d = px.scatter_3d(
    x=X_3d[:, 0],
    y=X_3d[:, 1],
    z=X_3d[:, 2],
    color=segmentation,
    category_orders={"color": ["A", "B", "C", "D"]},
)

fig_3d.update_traces(marker=dict(line=dict(width=0.5, color="white")))

for trace in fig_2d.data:
    fig.add_trace(trace, row=1, col=1)

for trace in fig_3d.data:
    fig.add_trace(trace, row=1, col=2)

fig.update_xaxes(title_text="PC1", row=1, col=1)
fig.update_yaxes(title_text="PC2", row=1, col=1)

fig.update_layout(
    scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
    title_text="Visualização PCA 2D vs 3D",
)

fig.show()

In [36]:
labels_clusters = segmentation
kernels_to_test = ["linear", "rbf", "poly", "sigmoid"]

#Loop de Plotagem
for kernel_name in kernels_to_test:
    print(f"--- Gerando gráficos para Kernel: '{kernel_name}' ---")

    # Calcular KPCA 2D e 3D
    kpca_2d = KernelPCA(n_components=2, kernel=kernel_name, random_state=42)
    X_2d = kpca_2d.fit_transform(X_train_transformed)

    kpca_3d = KernelPCA(n_components=3, kernel=kernel_name, random_state=42)
    X_3d = kpca_3d.fit_transform(X_train_transformed)

    # Início do Seu Código-Template

    fig = make_subplots(
        rows=1,
        cols=2,
        subplot_titles=("KPCA com 2 Componentes", "KPCA com 3 Componentes"),
        specs=[[{"type": "xy"}, {"type": "scene"}]],
    )

    # Gráfico 2D
    fig_2d = px.scatter(
        x=X_2d[:, 0],
        y=X_2d[:, 1],
        color=labels_clusters,
        category_orders={
            "color": ["A", "B", "C", "D"]
        },
    )

    fig_2d.update_traces(
        marker=dict(
            line=dict(width=0.5, color="white")
        )
    )

    # Gráfico 3D
    fig_3d = px.scatter_3d(
        x=X_3d[:, 0],
        y=X_3d[:, 1],
        z=X_3d[:, 2],
        color=labels_clusters,
        category_orders={
            "color": ["A", "B", "C", "D"]
        },
    )

    fig_3d.update_traces(
        marker=dict(
            line=dict(width=0.5, color="white")
        )
    )

    # Adicionar os traces ao subplot
    for trace in fig_2d.data:
        fig.add_trace(trace, row=1, col=1)

    for trace in fig_3d.data:
        fig.add_trace(trace, row=1, col=2)

    # Atualizar eixos
    fig.update_xaxes(title_text="PC1", row=1, col=1)
    fig.update_yaxes(title_text="PC2", row=1, col=1)

    fig.update_layout(
        scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"),
        title_text=f"Visualização 2D vs 3D - Kernel: '{kernel_name}'",
        showlegend=True,
        legend_title_text="Cluster",
    )

    fig.show()

Output hidden; open in https://colab.research.google.com to view.

Gráficos como o da Variância Explicada pelo PCA, Kernel PCA e as visualizações 2D e 3D nos revelam que os dados não são linearmente separáveis. Sendo que o melhor kernel é o Sigmoid, pois explica bem a variância dos dados.

O único dilema é que os dados estão muito agrupados. Se rodássemos um DBSCAN, ele provavelmente veria a maior parte dos dados como um cluster gigante (vamos testar essa tese). Por isso, também vamos testar o K-Means, mesmo que ele nos dê 4 clusters bem diferentes dos segmentos originais e que assuma clusters de formato redondo, depois vamos compará-los.

# KPCA

In [37]:
# 2 Dimensões
kpca_2d = KernelPCA(n_components=2, kernel="sigmoid", random_state=42)
X_2d_train_fit = kpca_2d.fit(X_train_transformed)
X_2d_train = kpca_2d.transform(X_train_transformed)
X_2d_test = kpca_2d.transform(X_test_transformed)

# DBSCAN

Vamos utilizar o método do cotovelo para encontrar o melhor epsilon.

In [38]:
def knee_point(distances):
    """
    Encontra o ponto do cotovelo usando o método da distância perpendicular.
    Retorna o índice do ponto do cotovelo.
    """
    n_points = len(distances)
    x1, y1 = 0, distances[0]
    x2, y2 = n_points - 1, distances[-1]

    max_distance = 0
    knee_idx = 0

    for i in range(1, n_points - 1):
        x0, y0 = i, distances[i]
        numerator = abs((y2 - y1) * x0 - (x2 - x1) * y0 + x2 * y1 - y2 * x1)
        denominator = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
        distance = numerator / denominator
        if distance > max_distance:
            max_distance = distance
            knee_idx = i

    return knee_idx


def plot_k_distance_plotly(data, k=5):
    """
    Cria um gráfico k-distance interativo com o Plotly e destaca o ponto do cotovelo.
    """
    if isinstance(data, pd.DataFrame):
        data = data.values

    # Calcula distância para o k-ésimo vizinho mais próximo
    neighbors = NearestNeighbors(n_neighbors=k)
    neighbors_fit = neighbors.fit(data)
    distances_matrix, _ = neighbors_fit.kneighbors(data)

    # Pega a distância do k-ésimo vizinho (última coluna)
    distances = np.sort(distances_matrix[:, -1])

    # Encontra o ponto do cotovelo
    knee_idx = knee_point(distances)
    eps_suggestion = float(distances[knee_idx])

    # Cria gráfico Plotly
    fig = go.Figure()
    highlight_color = "#ff7f0e"

    # Linha principal
    fig.add_trace(
        go.Scatter(
            x=np.arange(len(distances)),
            y=distances,
            mode="lines",
            name=f"K-distance (k={k})",
            line=dict(width=2),
        )
    )

    # Ponto do cotovelo
    fig.add_trace(
        go.Scatter(
            x=[knee_idx],
            y=[eps_suggestion],
            mode="markers+text",
            name="Ponto do cotovelo",
            text=[f"ε = {eps_suggestion:.4f}"],
            textposition="top right",
            marker=dict(
                color=highlight_color, size=10, symbol="circle"
            ),
        )
    )

    # Linhas tracejadas horizontais e verticais
    fig.add_shape(
        type="line",
        x0=knee_idx,
        x1=knee_idx,
        y0=0,
        y1=eps_suggestion,
        line=dict(color=highlight_color, width=1.5, dash="dash"),
    )

    fig.add_shape(
        type="line",
        x0=0,
        x1=knee_idx,
        y0=eps_suggestion,
        y1=eps_suggestion,
        line=dict(color=highlight_color, width=1.5, dash="dash"),
    )

    # Layout do gráfico
    fig.update_layout(
        title=f"Gráfico K-distance (k={k}) — ε sugerido = {eps_suggestion:.4f}",
        xaxis_title="Pontos ordenados pela distância",
        yaxis_title=f"Distância ao {k}º vizinho mais próximo",
        legend=dict(yanchor="top", y=0.99, xanchor="right", x=0.99),
    )

    fig.show()

    return eps_suggestion, knee_idx, distances


# Exemplo de uso
eps_suggestion, knee_idx, distances = plot_k_distance_plotly(X_2d_train, k=5)

print(f"\nEpsilon sugerido: {eps_suggestion:.4f}")
print(f"Índice do ponto do cotovelo: {knee_idx}")
print(f"Distância média do {5}º vizinho: {np.mean(distances):.4f}")
print(f"Distância mediana do {5}º vizinho: {np.median(distances):.4f}")


Epsilon sugerido: 0.0115
Índice do ponto do cotovelo: 7427
Distância média do 5º vizinho: 0.0066
Distância mediana do 5º vizinho: 0.0053


Agora que temos o epsilon ideal pelo método do cotovelo, vamos criar o modelo DBSCAN.

In [39]:
# Seus parâmetros
eps_ideal = eps_suggestion
min_samples_ideal = 5

# Aplicar o DBSCAN nos dados 2D
dbscan = DBSCAN(eps=eps_ideal, min_samples=min_samples_ideal)
clusters = dbscan.fit_predict(X_2d_train)

# Preparar os dados para visualização
df_2d = pd.DataFrame(X_2d_train, columns=["PC1", "PC2"])
df_2d["cluster"] = clusters  # Adiciona os resultados do cluster
df_2d["cluster"] = df_2d["cluster"].astype(str)  # Converte para string para o Plotly

# Visualizar os clusters
print(f"Número de clusters encontrados: {len(np.unique(clusters)) - (1 if -1 in clusters else 0)}")
print(f"Número de outliers (ruído): {np.sum(clusters == -1)}")

fig_clusters = px.scatter(
    df_2d,
    x="PC1",
    y="PC2",
    color="cluster",
    title=f"Clusters DBSCAN (eps={eps_ideal}, min_samples={min_samples_ideal})",
    color_discrete_map={"-1": "grey"},
)

fig_clusters.update_traces(marker=dict(size=8, opacity=0.7))
fig_clusters.update_layout(
    xaxis_title="Componente Principal 1 (PC1)",
    yaxis_title="Componente Principal 2 (PC2)",
    legend_title="Cluster",
)
fig_clusters.show()

Número de clusters encontrados: 20
Número de outliers (ruído): 449


Como podemos ver, mesmo com o epsilon ideal o modelo entende que a grande maioria dos dados pertencem ao mesmo cluster. Isso seria contraprodutivo para a nossa análise.

# K-Means

Agora vamos ver como o K-Means lida com isso.

In [40]:
# Treinar o K-Means (FIT)
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)

kmeans.fit(X_2d_train)

# Gerar Rótulos (PREDICT) para AMBOS
clusters_treino = kmeans.predict(X_2d_train)
clusters_teste = kmeans.predict(X_2d_test)

# Plotar Gráfico de TREINO
print("Gerando gráfico para dados de TREINO...")
silhouette_treino = silhouette_score(X_2d_train, clusters_treino)

df_train_plot = pd.DataFrame(X_2d_train, columns=["PC1", "PC2"])
df_train_plot["cluster"] = clusters_treino.astype(str)

fig_train = px.scatter(
    df_train_plot,
    x="PC1",
    y="PC2",
    color="cluster",
    title=f"Clusters K-Means (k={n_clusters}) — Dados de TREINO<br>Silhouette={silhouette_treino:.3f}",
)
fig_train.update_traces(marker=dict(size=8, opacity=0.7))
fig_train.update_layout(
    xaxis_title="Componente Principal 1 (kPCA_1)",
    yaxis_title="Componente Principal 2 (kPCA_2)",
    legend_title="Cluster",
)
fig_train.show()

# Plotar Gráfico de TESTE
print("Gerando gráfico para dados de TESTE...")
silhouette_test = silhouette_score(X_2d_test, clusters_teste)

df_test_plot = pd.DataFrame(X_2d_test, columns=["PC1", "PC2"])
df_test_plot["cluster"] = clusters_teste.astype(str)

fig_test = px.scatter(
    df_test_plot,
    x="PC1",
    y="PC2",
    color="cluster",
    title=f"Clusters K-Means (k={n_clusters}) — Dados de TESTE<br>Silhouette={silhouette_test:.3f}",
)
fig_test.update_traces(marker=dict(size=8, opacity=0.7))
fig_test.update_layout(
    xaxis_title="Componente Principal 1 (kPCA_1)",
    yaxis_title="Componente Principal 2 (kPCA_2)",
    legend_title="Cluster",
)

Gerando gráfico para dados de TREINO...


Gerando gráfico para dados de TESTE...


E assim ficaram os clusters. Durante a pesquisa para esse projeto, resolvi testar ajustar e treinar o PCA em dimensões altas e depois plotar os resultados com duas dimensões. Os resultados foram interessantes, já que os clusters ficaram mais parecidos com os segmentos originais, mas creio que por causa da maldição da dimensionalidade, os modelos não foram suficientes. Testei com ambos DBSCAN e K-Means, mas como o gráfico do K-Means ficou mais simples de entender, vou deixa-lo aqui para curiosidade.

In [41]:
pca_10d = KernelPCA(n_components=10, kernel="sigmoid", random_state=42)  # 90% de variância explicada
X_10d_train = pca_10d.fit_transform(X_train_transformed)
X_10d_test = pca_10d.transform(X_test_transformed)


# Treinar o K-Means (FIT)
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=42)
kmeans.fit(X_10d_train)


# Gerar Rótulos (PREDICT)
clusters_treino = kmeans.predict(X_10d_train)
clusters_teste = kmeans.predict(X_10d_test)


# Plotar Gráfico de TREINO
print("Gerando gráfico para dados de TREINO...")

# Calcule o score nos dados de treino
silhouette_treino = silhouette_score(X_10d_train, clusters_treino)
silhouette_teste = silhouette_score(X_10d_test, clusters_teste)

# Usando os dados 2D para os eixos X e Y
df_train_plot = pd.DataFrame(X_2d_train, columns=["PC1", "PC2"])
# Usando os clusters do modelo 10D para a COR
df_train_plot["cluster"] = clusters_treino.astype(str)

fig_train = px.scatter(
    df_train_plot,
    x="PC1",
    y="PC2",
    color="cluster",
    title=f"Clusters K-Means (Treinado em 10D) — Dados de TREINO<br>Silhouette (em 10D)={silhouette_treino:.3f}",
)
# ... resto do seu código de plotagem ...
fig_train.show()

# --- 5. Plotar Gráfico de TESTE (Lógica idêntica) ---
print("Gerando gráfico para dados de TESTE...")

df_test_plot = pd.DataFrame(X_2d_test, columns=["PC1", "PC2"])  # Eixos 2D
df_test_plot["cluster"] = clusters_teste.astype(str)  # Cor do predict(X_3d_test)

fig_test = px.scatter(
    df_test_plot,
    x="PC1",
    y="PC2",
    color="cluster",
    title=f"Clusters K-Means (Treinado em 10D) — Dados de TESTE<br>Silhouette (em 10D)={silhouette_teste:.3f}",
)
# ... resto do seu código de plotagem ...
fig_test.show()

Gerando gráfico para dados de TREINO...


Gerando gráfico para dados de TESTE...


# Comparação Entre o Cluster Final e os Segmentos

In [42]:
# Comparando os clusters gerados com os dados originais
clusters_treino_series = pd.Series(clusters_treino, index=train.index, name='Cluster')

# Comparando com a série original 'Segmentation'
train = pd.concat([train, segmentation], axis=1)
df = pd.concat([train, clusters_treino_series], axis=1)
df['Cluster'] = df['Cluster'].map({0: 'A', 1: 'B', 2: 'C', 3: 'D'})

display(df)

,ID,Gender,Ever_Married,Age,Graduated,Profession,Work_Experience,Spending_Score,Family_Size,Var_1,Segmentation,Cluster
0,462809,Male,No,22,No,Healthcare,1.0,Low,4.0,Cat_4,D,B
1,462643,Female,Yes,38,Yes,Engineer,NaN,Average,3.0,Cat_4,A,C
2,466315,Female,Yes,67,Yes,Engineer,1.0,Low,1.0,Cat_6,B,D
3,461735,Male,Yes,67,Yes,Lawyer,0.0,High,2.0,Cat_6,B,C
4,462669,Female,Yes,40,Yes,Entertainment,NaN,High,6.0,Cat_6,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...
8063,464018,Male,No,22,No,NaN,0.0,Low,7.0,Cat_1,D,B
8064,464685,Male,No,35,No,Executive,3.0,Low,4.0,Cat_4,D,B
8065,465406,Female,No,33,Yes,Healthcare,1.0,Low,1.0,Cat_6,D,D
8066,467299,Female,No,27,Yes,Healthcare,1.0,Low,4.0,Cat_6,B,B


In [43]:
# Tabela cruzada
conf_matrix = pd.crosstab(df["Segmentation"], df["Cluster"])

fig = px.imshow(
    conf_matrix,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Matriz de Comparação Segmentos vs Clusters",
    labels=dict(x="Cluster", y="Segmento", color="Contagem")
)
fig.show()


Os resultados mostram uma diferença grande entre os segmentos originais e clusters do modelo. Clusters C e A foram os que mais ficaram similares.

In [44]:
fig = px.box(
    train,
    x="Segmentation",
    y="Age",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    title="Idade por Segmento",
)

fig.update_layout(
    xaxis_title="Segmento", yaxis_title="Idade", yaxis_dtick=10, showlegend=False
)

fig.show()

fig = px.box(
    df,
    x="Cluster",
    y="Age",
    color="Cluster",
    category_orders={"Cluster": ["A", "B", "C", "D"]},
    title="Idade por Cluster",
)

fig.update_layout(
    xaxis_title="Cluster", yaxis_title="Idade", yaxis_dtick=10, showlegend=False
)

fig.show()

As idades ficaram muito diferentes também, especialmente o Cluster B que ficou composto de pessoas mais novas.

In [45]:
fig = px.box(
    train,
    x="Segmentation",
    y="Family_Size",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    title="Tamanho da Família por Segmento",
)

fig.update_layout(
    xaxis_title="Segmento",
    yaxis_title="Tamanho da Família",
    yaxis_dtick=1,
    showlegend=False,
)

fig.show()

fig = px.box(
    df,
    x="Cluster",
    y="Family_Size",
    color="Cluster",
    category_orders={"Cluster": ["A", "B", "C", "D"]},
    title="Tamanho da Família por Cluster",
)

fig.update_layout(
    xaxis_title="Cluster",
    yaxis_title="Tamanho da Família",
    yaxis_dtick=1,
    showlegend=False,
)

fig.show()

As pessoas que estavam nos segmentos A e D foram para o Cluster B, no quesito do tamanho da família.

In [46]:
fig = px.box(
    train,
    x="Segmentation",
    y="Work_Experience",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    title="Experiência de Trabalho por Segmento",
)

fig.update_layout(
    xaxis_title="Segmento",
    yaxis_title="Experiência de Trabalho (anos)",
    yaxis_dtick=1,
    showlegend=False,
)

fig.show()

fig = px.box(
    df,
    x="Cluster",
    y="Work_Experience",
    color="Cluster",
    category_orders={"Cluster": ["A", "B", "C", "D"]},
    title="Experiência de Trabalho por Cluster",
)

fig.update_layout(
    xaxis_title="Cluster",
    yaxis_title="Experiência de Trabalho (anos)",
    yaxis_dtick=1,
    showlegend=False,
)

fig.show()

As pessoas com mais experiência de trabalho foram em sua maioria para o cluster D.

In [47]:
fig = px.histogram(
    train,
    x="Spending_Score",
    color="Segmentation",
    barmode="group",
    category_orders={
        "Spending_Score": ["Low", "Average", "High"],
        "Segmentation": ["A", "B", "C", "D"],
    },
    title="Score de Gastos por Segmento",
    text_auto=True,
    labels={"Spending_Score": "Score de Gastos", "Segmentation": "Segmento"},
)

fig.update_layout(
    xaxis_title="Score de Gastos",
    yaxis_title="Contagem",
)

fig.show()

fig = px.histogram(
    df,
    x="Spending_Score",
    color="Cluster",
    barmode="group",
    category_orders={
        "Spending_Score": ["Low", "Average", "High"],
        "Cluster": ["A", "B", "C", "D"],
    },
    title="Score de Gastos por Cluster",
    text_auto=True,
    labels={"Spending_Score": "Score de Gastos", "Cluster": "Cluster"},
)

fig.update_layout(
    xaxis_title="Score de Gastos",
    yaxis_title="Contagem",
)

fig.show()

As pessoas que gastam de forma moderada e alta ficaram no cluster C, enquanto a grande maioria dos outros clusters ficaram na categoria baixa.

In [48]:
fig = px.histogram(
    train,
    x="Gender",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    barmode="group",
    title="Gênero por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)
fig.update_layout(xaxis_title="Segmento", yaxis_title="Gênero")
fig.show()

fig = px.histogram(
    df,
    x="Gender",
    color="Cluster",
    category_orders={"Cluster": ["A", "B", "C", "D"]},
    barmode="group",
    title="Gênero por Cluster",
    text_auto=True,
    labels={"Cluster": "Cluster"},
)
fig.update_layout(xaxis_title="Cluster", yaxis_title="Gênero")
fig.show()

As distribuições também mudaram um pouco.

In [49]:
fig = px.histogram(
    train,
    x="Profession",
    color="Segmentation",
    category_orders={
        "Segmentation": ["A", "B", "C", "D"],
    },
    barmode="group",
    title="Profissão por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)

fig.update_layout(
    xaxis_title="Profissão",
    yaxis_title="Contagem",
    xaxis_categoryorder="total descending",
)

fig.show()

fig = px.histogram(
    df,
    x="Profession",
    color="Cluster",
    category_orders={
        "Cluster": ["A", "B", "C", "D"],
    },
    barmode="group",
    title="Profissão por Cluster",
    text_auto=True,
    labels={"Cluster": "Cluster"},
)

fig.update_layout(
    xaxis_title="Profissão",
    yaxis_title="Contagem",
    xaxis_categoryorder="total descending",
)

fig.show()

Ocorreu uma grande mudança no Segmento D para o cluster B na área da saúde, também houve uma mudança significativa na área dos artistas.

In [50]:
profession_order = train["Profession"].value_counts().index.tolist()

fig = px.density_heatmap(
    train,
    y="Profession",
    x="Spending_Score",
    facet_col="Segmentation",
    category_orders={
        "Segmentation": ["A", "B", "C", "D"],
        "Spending_Score": ["Low", "Average", "High"],
        "Profession": profession_order,
    },
    title="Densidade de Score de Gastos por Profissão e Segmento",
    color_continuous_scale="Viridis",
    text_auto=True,
)

fig.update_xaxes(title_text="Score de Gastos")

fig.update_layout(yaxis_title="Profissão")

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.update_layout(coloraxis_colorbar_title_text="Contagem")

fig.show()

fig = px.density_heatmap(
    df,
    y="Profession",
    x="Spending_Score",
    facet_col="Cluster",
    category_orders={
        "Cluster": ["A", "B", "C", "D"],
        "Spending_Score": ["Low", "Average", "High"],
        "Profession": profession_order,
    },
    title="Densidade de Score de Gastos por Profissão e Cluster",
    color_continuous_scale="Viridis",
    text_auto=True,
)

fig.update_xaxes(title_text="Score de Gastos")

fig.update_layout(yaxis_title="Profissão")

fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))

fig.update_layout(coloraxis_colorbar_title_text="Contagem")

fig.show()

Como visto antes as pessoas que gastam de forma moderada e alta ficaram no cluster C. Detalhe importante que não existe ninguém que gasta além de baixo no cluster A.

In [51]:
fig = px.histogram(
    train,
    x="Graduated",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    barmode="group",
    title="Graduação por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)
fig.update_layout(xaxis_title="Graduado?", yaxis_title="Contagem")
fig.show()

fig = px.histogram(
    df,
    x="Graduated",
    color="Cluster",
    category_orders={"Cluster": ["A", "B", "C", "D"]},
    barmode="group",
    title="Graduação por Cluster",
    text_auto=True,
    labels={"Cluster": "Cluster"},
)
fig.update_layout(xaxis_title="Graduado?", yaxis_title="Contagem")
fig.show()

Houve grande mudança no Segmento B para o Cluster B.

In [52]:
fig = px.histogram(
    train,
    x="Ever_Married",
    color="Segmentation",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    barmode="group",
    title="Estado Civil por Segmento",
    text_auto=True,
    labels={"Segmentation": "Segmento"},
)
fig.update_layout(xaxis_title="Casado?", yaxis_title="Contagem")
fig.show()

fig = px.histogram(
    df,
    x="Ever_Married",
    color="Cluster",
    category_orders={"Cluster": ["A", "B", "C", "D"]},
    barmode="group",
    title="Estado Civil por Cluster",
    text_auto=True,
    labels={"Cluster": "Cluster"},
)
fig.update_layout(xaxis_title="Casado?", yaxis_title="Contagem")
fig.show()

As pessoas do cluster B ficaram no grupo dos não casados e as do cluster C ficaram no grupo dos casados.

In [53]:
fig = px.density_heatmap(
    train,
    x="Var_1",
    y="Segmentation",
    text_auto=True,
    title="Relação entre Var_1 e Segmentação",
    category_orders={"Segmentation": ["A", "B", "C", "D"]},
    color_continuous_scale="Viridis",
)

fig.update_layout(
    xaxis_title="Var_1",
    yaxis_title="Segmento",
    coloraxis_colorbar_title_text="Contagem",
    xaxis_categoryorder="category ascending",
)

fig.show()

fig = px.density_heatmap(
    df,
    x="Var_1",
    y="Cluster",
    text_auto=True,
    title="Relação entre Var_1 e Cluster",
    category_orders={"Cluster": ["A", "B", "C", "D"]},
    color_continuous_scale="Viridis",
)

fig.update_layout(
    xaxis_title="Var_1",
    yaxis_title="Cluster",
    coloraxis_colorbar_title_text="Contagem",
    xaxis_categoryorder="category ascending",
)

fig.show()

Essas relações ficaram parecidas, com destaque para um aumento do cluster C na Cat_6.

# MlFlow

In [59]:
import mlflow
import mlflow.sklearn

mlflow.set_experiment("/Users/luizg.dev@gmail.com/experimentos/pca_dbscan_kmeans")

# === Kernel PCA ===
with mlflow.start_run(run_name="KernelPCA_DBSCAN_KMeans"):

    # Parâmetros principais
    kernel = "sigmoid"
    n_components = 2
    random_state = 42

    mlflow.log_params({
        "kernel": kernel,
        "n_components": n_components,
        "random_state": random_state
    })

    kpca = KernelPCA(n_components=n_components, kernel=kernel, random_state=random_state)
    X_2d_train_fit = kpca.fit(X_train_transformed)
    X_2d_train = kpca.transform(X_train_transformed)
    X_2d_test = kpca.transform(X_test_transformed)

    # === DBSCAN ===
    def knee_point(distances):
        n_points = len(distances)
        x1, y1 = 0, distances[0]
        x2, y2 = n_points - 1, distances[-1]
        max_distance = 0
        knee_idx = 0
        for i in range(1, n_points - 1):
            x0, y0 = i, distances[i]
            numerator = abs((y2 - y1) * x0 - (x2 - x1) * y0 + x2 * y1 - y2 * x1)
            denominator = np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)
            distance = numerator / denominator
            if distance > max_distance:
                max_distance = distance
                knee_idx = i
        return knee_idx

    neighbors = NearestNeighbors(n_neighbors=5)
    distances_matrix, _ = neighbors.fit(X_2d_train).kneighbors(X_2d_train)
    distances = np.sort(distances_matrix[:, -1])
    knee_idx = knee_point(distances)
    eps_ideal = float(distances[knee_idx])
    min_samples_ideal = 5

    mlflow.log_params({
        "eps": eps_ideal,
        "min_samples": min_samples_ideal
    })

    dbscan = DBSCAN(eps=eps_ideal, min_samples=min_samples_ideal)
    clusters_dbscan = dbscan.fit_predict(X_2d_train)

    n_clusters_dbscan = len(np.unique(clusters_dbscan)) - (1 if -1 in clusters_dbscan else 0)
    n_outliers = np.sum(clusters_dbscan == -1)

    mlflow.log_metrics({
        "dbscan_num_clusters": n_clusters_dbscan,
        "dbscan_num_outliers": n_outliers
    })

    # === KMeans ===
    n_clusters = 4
    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    kmeans.fit(X_2d_train)

    clusters_treino = kmeans.predict(X_2d_train)
    clusters_teste = kmeans.predict(X_2d_test)

    silhouette_treino = silhouette_score(X_2d_train, clusters_treino)
    silhouette_teste = silhouette_score(X_2d_test, clusters_teste)

    mlflow.log_metrics({
        "silhouette_train": silhouette_treino,
        "silhouette_test": silhouette_teste
    })

    # === Salvar modelo (opcional) ===
    mlflow.sklearn.log_model(kmeans, artifact_path="kmeans_model")
    mlflow.sklearn.log_model(dbscan, artifact_path="dbscan_model")

    print(f"KMeans silhouette (train): {silhouette_treino:.4f}")
    print(f"KMeans silhouette (test):  {silhouette_teste:.4f}")
    print(f"DBSCAN clusters: {n_clusters_dbscan}, outliers: {n_outliers}")


2025/11/02 21:42:20 INFO mlflow.tracking.fluent: Experiment with name '/Users/luizg.dev@gmail.com/experimentos/pca_dbscan_kmeans' does not exist. Creating a new experiment.
2025/11/02 21:42:48 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/02 21:43:01 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
2025/11/02 21:43:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/02 21:43:01 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
2025/11/02 21:43:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


KMeans silhouette (train): 0.4075
KMeans silhouette (test):  0.4112
DBSCAN clusters: 20, outliers: 449


# Conclusão

A análise e modelagem realizadas neste notebook demonstram os desafios na segmentação de clientes quando a estrutura dos dados não é claramente linear. Embora o K-Means tenha conseguido dividir os dados em 4 clusters, a dissimilaridade entre esses clusters e os segmentos originais (conforme revelado pela matriz de confusão e comparação das distribuições das variáveis) indica que o modelo não conseguiu replicar a segmentação existente.

O DBSCAN não se mostrou adequado para este problema específico devido à sua tendência em agrupar a maioria dos dados em um único cluster quando os dados não possuem densidades bem separadas.

O Kernel PCA com kernel 'sigmoid' foi essencial para reduzir a dimensionalidade mantendo a não linearidade dos dados, o que é crucial para algoritmos de cluster que operam em espaços de menor dimensão.

Para aprimorar a segmentação, futuras etapas podem incluir:
- **Exploração de outros algoritmos de clustering**: Testar algoritmos como o Agglomerative Clustering, Spectral Clustering, ou Gaussian Mixture Models.
- **Otimização de hiperparâmetros**: Utilizar técnicas como Grid Search ou Randomized Search com validação cruzada para encontrar os melhores parâmetros para o K-Means ou outros algoritmos testados.
- **Engenharia de features mais avançada**: Criar novas features que possam capturar melhor as características dos clientes e as diferenças entre os segmentos.
- **Validação externa**: Se disponível, validar os clusters com dados externos ou conhecimento de domínio para avaliar sua relevância prática.

A utilização do MLflow foi fundamental para organizar e rastrear os experimentos, facilitando a comparação dos resultados de diferentes abordagens e a reprodutibilidade do trabalho. Embora o K-Means não tenha replicado os segmentos originais, os clusters gerados podem ainda fornecer insights valiosos para estratégias de marketing ou outras ações direcionadas, dependendo dos perfis de clientes que emergiram em cada cluster. A análise das características de cada cluster gerado pelo K-Means (conforme a comparação detalhada das variáveis) é um passo crucial para entender esses novos segmentos.